# @zz-windfm-test/quartz-solar-forecast

Open Climate Fix's open-source 48-hour solar PV forecast from weather predictions and a boosted-tree model.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9). The last code cell prints the endpoint address.


In [ ]:
PRIMITIVE = "@zz-windfm-test/quartz-solar-forecast"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/zz-windfm-test/quartz-solar-forecast/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import importlib.metadata as md, os, subprocess, sys
from pathlib import Path

print(f"Python {sys.version.split()[0]}")

# PY is the interpreter the later cells install into and serve from. It is Colab's own unless
# this primitive asks for another Python (WANT, empty when it does not say): then a virtual
# environment on that Python is made with uv, which fetches the interpreter itself.
PY = sys.executable
WANT = "3.11"
if WANT and not sys.version.startswith(WANT + "."):
    VENV = "/content/venv"
    try:
        for step in (
            [sys.executable, "-m", "pip", "install", "-q", "uv"],
            [sys.executable, "-m", "uv", "venv", "--seed", "--python", WANT, VENV],
        ):
            subprocess.run(step, check=True, capture_output=True, text=True)
        PY = VENV + "/bin/python"
        # Colab sets PYTHONPATH to its own packages, built for its own Python.
        os.environ.pop("PYTHONPATH", None)
        print(f"This primitive asks for Python {WANT}: using it in {VENV}")
    except subprocess.CalledProcessError as exc:
        print(f"Could not set up Python {WANT}; staying on {sys.version.split()[0]}:")
        print((exc.stderr or "")[-600:])

# The file nearest the root first: a requirements.txt under docs/ or tests/ is not this
# primitive's, and rglob alone would pick whichever it met first.
found = sorted(
    Path(WORKDIR).rglob("requirements.txt"),
    key=lambda p: (len(p.relative_to(WORKDIR).parts), str(p)),
)
if not found:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
    if PY != sys.executable:
        subprocess.run([PY, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], capture_output=True)
else:
    req = found[0]
    print(f"Installing {req.relative_to(WORKDIR)} …")
    if len(found) > 1:
        others = ", ".join(str(p.relative_to(WORKDIR)) for p in found[1:4])
        print(f"Not installed (only one file is): {others}")

    def versions():
        return {d.metadata["Name"].lower(): d.version for d in md.distributions() if d.metadata["Name"]}

    def install():
        # The endpoint needs FastAPI and uvicorn, which Colab ships and a fresh environment does
        # not. Asked for in the same install, pip picks versions that fit the primitive's pins.
        serve = ["fastapi", "uvicorn"] if PY != sys.executable else []
        return subprocess.run(
            [PY, "-m", "pip", "install", "-q", "-r", str(req), *serve],
            capture_output=True, text=True,
        )

    before = versions()
    done = install()
    if done.returncode != 0 and PY != sys.executable:
        # Keep the requested Python only if it works: fall back to Colab's, and say so.
        print(f"pip failed on Python {WANT}; trying Colab's {sys.version.split()[0]} instead.")
        PY = sys.executable
        done = install()
    if done.returncode == 0:
        print("done")
        after = versions()
        # An install can replace what Colab ships; say so, so a later surprise has a cause.
        for name in sorted(n for n in before if n in after and before[n] != after[n])[:12]:
            print(f"  changed: {name} {before[name]} -> {after[name]}")
    else:
        print(
            "pip failed. pip installs the whole file or none of it, so one package that "
            "cannot be built on this Python stops all of it:"
        )
        print(done.stderr[-2000:])


In [ ]:
import os, re, subprocess, sys, time, urllib.request

PY = globals().get("PY") or sys.executable  # the install cell's choice, if this session has it
PORT = 8011
MODULE = "powerai_endpoint"
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe: a previous run leaves a server and a tunnel
# alive, and the next one would otherwise pass the health probe anyway because the
# *stale* server answers it — a stale process serving stale code, reported as fresh.
for pattern in ("cloudflared tunnel", f"uvicorn {MODULE}:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# --app-dir puts WORKDIR on the import path for this one process, so the archive
# need not be pip-installed (and may not even be a package) to be served.
# A src/ layout keeps its packages one folder down; both folders go on the path.
env = dict(os.environ, PYTHONPATH=os.pathsep.join(
    [WORKDIR, os.path.join(WORKDIR, "src"), os.environ.get("PYTHONPATH", "")]
))
# The server's own output goes to a file, not nowhere: a file that fails on import, or hides an
# error behind a plain "internal" reply, leaves its cause here.
LOG = os.path.join(os.path.dirname(WORKDIR), "endpoint.log")
server = subprocess.Popen(
    [PY, "-m", "uvicorn", f"{MODULE}:app", "--app-dir", WORKDIR,
     "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env,
)

healthy = False
for _ in range(90):
    if server.poll() is not None:
        break  # it exited: nothing to wait for
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        healthy = True
        break
    except Exception:
        time.sleep(1)
if not healthy:
    try:
        output = open(LOG).read()[-3000:]
    except OSError:
        output = ""
    raise RuntimeError(
        f"{MODULE}.py did not come up as a healthy server on :{PORT}. Its output:\n{output}\n"
        "This is the archive's own code, not the Hub's."
    )
print(f"endpoint up on :{PORT}")
print(f"Server log: {LOG}  (after a failed call, run  !tail -n 40 {LOG}  to see why)")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + "/run"
print()
print("Endpoint address:")
print("   ", ENDPOINT_URL)

PASTE_LINE = f"Run {PRIMITIVE} at {ENDPOINT_URL}"
print()
print("Paste this line into the agent chat:")
print("   ", PASTE_LINE)
try:
    import json
    from IPython.display import HTML, display

    _line = json.dumps(PASTE_LINE).replace("<", chr(92) + "u003c")
    display(HTML(
        '<button id="powerai-copy" style="padding:6px 12px;cursor:pointer">Copy the line</button>'
        ' <span id="powerai-copied"></span>'
        "<script>(function () {"
        " var line = " + _line + ";"
        " var button = document.getElementById('powerai-copy');"
        " var note = document.getElementById('powerai-copied');"
        " function fallback() {"
        "  var box = document.createElement('textarea'); box.value = line;"
        "  document.body.appendChild(box); box.select();"
        "  try { document.execCommand('copy'); note.textContent = ' Copied.'; }"
        "  catch (e) { note.textContent = ' Select the line above and copy it.'; }"
        "  document.body.removeChild(box);"
        " }"
        " button.onclick = function () {"
        "  if (navigator.clipboard && navigator.clipboard.writeText) {"
        "   navigator.clipboard.writeText(line).then(function () { note.textContent = ' Copied.'; }, fallback);"
        "  } else { fallback(); }"
        " };"
        "})();</script>"
    ))
except Exception:
    pass  # the plain line above is enough


Copy the line printed above (use the Copy button, or select it) and paste it into the agent chat on [the Agent tab](https://hub.powerai.ai/zz-windfm-test/quartz-solar-forecast/agent).